# Training the "Hey Claude" wake-word model (version 2)

Trains a microWakeWord model for the phrase "Hey Claude", for the Hey Claude
device firmware. Adapted from microWakeWord's `basic_training_notebook.ipynb`
(Apache License 2.0), with synthetic voices chosen for a range of accents,
sound-alike phrases as negatives, recordings of real voices, a test on real
voices, and progress saved to Google Drive. Version 1 of this notebook,
which trained runs 1 and 2, is kept beside it as the record of how they
were made.

How to run it, what each section does, and what to do with the result are
described in `docs/WAKEWORD-TRAINING.md` in the repository. In short:
Runtime → Change runtime type → a GPU. Run **Install**, then Runtime →
Restart session. Run every section from **Settings** to **Negative
examples**. Restart the session again, then run **Settings**, **Helpers**,
**Train**, **Test on real voices** and **Keep results**.

**Licence of the result:** the background-noise datasets downloaded below carry
mixed licences, some non-commercial, and the L2-ARCTIC voice is trained on
recordings licensed CC BY-NC 4.0. A model trained with them is for
non-commercial use.

**Privacy:** recordings of real voices are personal data. They are read from
your own Drive and converted on the session's own disk. The players in
**Recordings** and **Augmentation** embed the audio they play in the
notebook's outputs, which Colab saves with the notebook in your Drive:
before saving a copy anywhere else or sharing it, use Edit → Clear all
outputs. Never add recordings, or this notebook with its outputs, to the
repository.

## Install

In [ ]:
# Installs the tools. microWakeWord, the sample generator and the two audio
# libraries are pinned; the rest is whatever Colab provides, and each run
# records the full list in its settings record (see Keep results).
# When this finishes, use Runtime -> Restart session, then run Settings.
# The last install fetches PyTorch and can take 5-15 minutes; it shows its
# progress so a long wait can be told apart from a stall.

MICROWAKEWORD_COMMIT = "4665173cd35f1cff9a61e06fc427f124766c488e"

# audio-metadata from a fork that unpins `attrs`, as microWakeWord's own notebook does
!pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
!git clone -q https://github.com/kahrendt/microWakeWord && cd microWakeWord && git checkout -q {MICROWAKEWORD_COMMIT}
!cd /content && pip install -q -e ./microWakeWord
!pip install piper-sample-generator==3.2.0
# The 3.2.0 package leaves out `piper_train`, which the US English generator
# voice needs to load. It is plain PyTorch code; the source at the same
# version supplies it (made importable in Helpers).
!git clone -q --depth 1 --branch v3.2.0 https://github.com/rhasspy/piper-sample-generator /content/piper-sample-generator
# The generator pins audiomentations to 0.33.0, which lacks AddColorNoise that
# microWakeWord's augmentation uses. Only the generator's optional augment tool
# uses audiomentations, and this notebook does not run it, so the newer version
# is installed over the pin. pip warns about the conflict; that is expected.
!pip install -q audiomentations==0.43.1

## Settings

In [ ]:
# Everything a run might change, and nothing else: the functions the later
# sections share are in Helpers. Run this, then Helpers, first in every
# session and again after every restart. Every value here is written into
# the run's settings record (see Keep results).

NOTEBOOK = 'hey_claude_v2'

# --- Where things are kept ---
# Drive keeps what is slow to make or must survive a disconnect; the session's
# own disk holds what is quick to rebuild.
KEEP = '/content/drive/MyDrive/hey_claude_wakeword'
SAMPLES = f'{KEEP}/samples'          # synthetic samples, reused by later runs
RECORDINGS = f'{KEEP}/recordings'    # real voices: personal data, Drive only

# Each run keeps its checkpoints, settings and results under its own name, so
# a new run starts fresh and an earlier one stays for comparison. Run history
# is in docs/WAKEWORD-TRAINING.md.
RUN = 'run3'
TRAIN_DIR = f'{KEEP}/trained_models/hey_claude_{RUN}'
FEATURES = f'features/{RUN}'          # session disk
CLIPS_16K = 'recordings_16k'          # converted recordings, session disk

# --- The phrase and its sound-alikes ---
# Spelled for each kind of voice. English voices read it as written; the
# Russian voices read a transliteration, which gives "Hey Claude" with a
# Russian accent. (Piper's Hebrew voice would add another accent, but it
# needs a newer Piper than the sample generator supports.)
PHRASE_EN = 'hey claude'
PHRASE_RU = 'хэй клод'

# Phrases that sound like the wake phrase but must not trigger it. "claude" and
# "hey" alone matter most: people near the device will talk about Claude.
SOUND_ALIKES = [
    'claude', 'hey', 'hey cloud', 'hey clyde', 'hey claw', 'hey clod', 'hey lord',
    'hey maud', 'hey claudia', 'okay claude', 'hi claude', 'say claude', 'hey cod',
    'hey clause', 'hey glad', 'they clawed', 'a cloud', 'hey jarvis', 'claude said',
    'ask claude',
]

# --- Synthetic voices and samples ---
# The US English "generator" mixes 904 LibriTTS-R speakers; the others are
# ordinary Piper voices that add accents the generator lacks.
GENERATOR_VOICE = 'voices/en_US-libritts_r-medium.pt'
VOICES = {
    # 24 speakers of English as a second language (L2-ARCTIC): Arabic, Mandarin,
    # Hindi, Korean, Spanish and Vietnamese first languages
    'l2arctic': 'en/en_US/l2arctic/medium/en_US-l2arctic-medium',
    # 109 speakers of British and other English accents (VCTK)
    'vctk': 'en/en_GB/vctk/medium/en_GB-vctk-medium',
    'ru_denis': 'ru/ru_RU/denis/medium/ru_RU-denis-medium',
    'ru_dmitri': 'ru/ru_RU/dmitri/medium/ru_RU-dmitri-medium',
    'ru_irina': 'ru/ru_RU/irina/medium/ru_RU-irina-medium',
    'ru_ruslan': 'ru/ru_RU/ruslan/medium/ru_RU-ruslan-medium',
}
def voice(name):
    return 'voices/' + VOICES[name].split('/')[-1] + '.onnx'

# Speaking speeds, from fast (0.8) to slow (1.3)
SPEEDS = '0.8 0.9 1.0 1.1 1.2 1.3'

# The sample sets, each a folder under SAMPLES. The generator runs on the GPU
# in batches; --max-speakers 800 avoids the last LibriTTS speakers, which have
# little training data and produce artefacts. Accented voices: some speakers
# mispronounce the phrase, so each is kept to a small share of the samples;
# enough for the accents, not enough to teach a sloppy version of the phrase.
SAMPLE_SETS = {
    'positive/generator': dict(phrases=PHRASE_EN, models=[GENERATOR_VOICE], count=15000,
                               batch=100, extra='--max-speakers 800'),
    'positive/l2arctic': dict(phrases=PHRASE_EN, models=[voice('l2arctic')], count=1000),
    'positive/vctk': dict(phrases=PHRASE_EN, models=[voice('vctk')], count=1000),
    'positive/ru': dict(phrases=PHRASE_RU, count=600,
                        models=[voice(v) for v in ('ru_denis', 'ru_dmitri', 'ru_irina', 'ru_ruslan')]),
    'negative/generator': dict(phrases=SOUND_ALIKES, models=[GENERATOR_VOICE], count=4000,
                               batch=100, extra='--max-speakers 800'),
    'negative/l2arctic': dict(phrases=SOUND_ALIKES, models=[voice('l2arctic')], count=1000),
}

# --- Augmentation: how each sample is roughened before training ---
# Mixed with background sound, given room echo, and varied in level and tone,
# so the model hears the phrase as a microphone across a room would. From
# microWakeWord's notebook, except that the background is never louder than
# the voice (0 to 15 dB below it, against -5 to 10 dB there): run 1 learned
# largely from clips where the phrase was barely audible.
AUGMENT = {
    'augmentation_duration_s': 3.2,
    'augmentation_probabilities': {
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    'background_min_snr_db': 0,
    'background_max_snr_db': 15,
    'min_jitter_s': 0.195,
    'max_jitter_s': 0.205,
}

# Background sound and room echoes for augmentation (see Background sound)
BACKGROUND_SOURCES = {
    'mit_rirs': dict(dataset='davidscripka/MIT_environmental_impulse_responses'),
    # One of the 38 parts of AudioSet's balanced set, about 690 MB, as
    # microWakeWord's notebook uses
    'audioset_16k': dict(dataset='agkphysics/AudioSet', data_files='data/bal_train/09.parquet'),
    'fma_16k': dict(url='https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip'),
}

# --- Recordings of real voices (see docs/WAKEWORD-RECORDING.md) ---
RECORDING = {
    # Each training recording is used as this many differently augmented
    # copies, without added room echo: recordings made across a room
    # already carry the real one
    'copies': 20,
    # The share of wake-phrase training examples drawn from recordings, and
    # the same for the sound-alikes: enough to matter, not so much that the
    # model memorises a hundred clips
    'share': 0.2,
}
# Finding the speech in a recording: speech is sound at least above_db over
# the recording's background level, and phrases in a long take are separated
# by at least gap_s of quiet. Each phrase is kept with pad_s either side:
# with the 0.2 s microWakeWord adds at the end (AUGMENT's jitter), a
# training clip ends about 0.4 s after the speech, which is when the model
# reaches its full score. Sounds shorter than min_phrase_s are clicks. A recording shorter than
# min_take_s holds one phrase, whatever its pauses. Frames quieter than
# silence_db are digital silence and do not count towards the background.
# above_db was 12 until run 3's recordings: with a kitchen running, a voice
# peaks only about 13 dB over the background, and 6 of 100 recordings were
# dropped as holding no speech. At 9, none are.
CUT = dict(above_db=9, gap_s=0.7, pad_s=0.2, min_phrase_s=0.3, min_take_s=5.0, silence_db=6)
AUDIO_TYPES = ['.wav', '.mp3', '.m4a', '.aac', '.flac', '.ogg', '.opus', '.3gp', '.amr', '.webm']

# --- Negative examples (see Negative examples) ---
NEGATIVE_URL = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main'
# Sets used for training only; their validation and testing parts are set
# aside to save memory
NEGATIVE_TRAINING_ONLY = ['dinner_party', 'no_speech', 'speech']
# The share of the dinner-party recording's validation part used while
# training
VALIDATION_SHARE = 0.15

# --- Training ---
# From microWakeWord's notebook, with these changes: the sound-alike phrases
# are added as negatives, weighted up because they are the likeliest false
# triggers; the best checkpoint is chosen to stay under TARGET_FAPH false
# detections per hour (the project's target, KNOWN-ISSUES R1) and then catch
# as many wake phrases as possible; and training is longer, 20,000 steps and
# then 10,000 at a tenth of the learning rate to settle. Settings given as
# one-entry lists apply to both phases.
TARGET_FAPH = 0.5
TRAINING = {
    'training_steps': [20000, 10000],
    'learning_rates': [0.001, 0.0001],
    'positive_class_weight': [1],
    'negative_class_weight': [20],
    'batch_size': 128,
    'time_mask_max_size': [0],
    'time_mask_count': [0],
    'freq_mask_max_size': [0],
    'freq_mask_count': [0],
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
}
# How often each feature set is drawn from, whatever its size, and how much
# its mistakes count (see Train)
SET_WEIGHTS = {'positive': 2.0, 'sound_alikes': 5.0, 'speech': 10.0, 'dinner_party': 10.0,
               'no_speech': 5.0}
SOUND_ALIKE_PENALTY = 2.0
# The model's shape, which matches the stock models the firmware was built
# with (stride 3)
MODEL_ARGS = ['mixednet', '--pointwise_filters', '64,64,64,64', '--repeat_in_block', '1, 1, 1, 1',
              '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]', '--residual_connection', '0,0,0,0',
              '--first_conv_filters', '32', '--first_conv_kernel_size', '5', '--stride', '3']

# --- The firmware's view of the model ---
STEP_MS = 10        # feature step; the manifest's feature_step_size
WINDOW = 5          # scores averaged per decision; the manifest's sliding_window_size
# Working memory for the model on the device, for the manifest. Measured for
# the MODEL_ARGS shape (run 2: 25,548 bytes used); change both together.
ARENA = 26000

## Helpers

In [ ]:
# The functions the later sections share. Run after Settings, in every
# session and after every restart.

import hashlib
import importlib.util
import json
import os
import re
import shutil
import subprocess
from pathlib import Path

import numpy as np
import scipy.io.wavfile
from google.colab import drive

# Install runs quietly, so a failed install would otherwise surface only in
# Augmentation. Rerun its microWakeWord line without -q to see why.
if importlib.util.find_spec('microwakeword') is None:
    raise RuntimeError('microWakeWord is not installed: run Install again, then Runtime -> Restart session')
if shutil.which('ffmpeg') is None:
    raise RuntimeError('ffmpeg is missing; Colab normally provides it')

drive.mount('/content/drive')
# Lets the sample generator, run as a separate program, find `piper_train`
# (see Install)
os.environ['PYTHONPATH'] = '/content/piper-sample-generator:' + os.environ.get('PYTHONPATH', '')
os.chdir('/content')
os.makedirs(KEEP, exist_ok=True)


# --- Skipping finished steps ---
# Every step that makes something writes a marker holding a fingerprint of
# the settings it was made with. A step is skipped only when its marker
# matches; a step stopped part-way, or one whose settings have changed, is
# redone from the start.

def fingerprint(settings):
    """A short hash of a step's settings, stored in its marker."""
    text = json.dumps(settings, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(text.encode()).hexdigest()[:16]


def step_done(path, settings):
    """True if the step that makes `path` finished with these settings."""
    marker = Path(path, '.done')
    return marker.exists() and marker.read_text().strip() == fingerprint(settings)


def mark_done(path, settings):
    Path(path, '.done').write_text(fingerprint(settings))


def fresh(path):
    """Empties `path` before a step makes it. The marker goes first, so a
    delete stopped part-way never leaves an old marker over what remains."""
    Path(path, '.done').unlink(missing_ok=True)
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path)


def markers(root):
    """The markers of every finished step under `root`: a fingerprint of a
    step's inputs, so that what is built from them is redone when they
    change."""
    return sorted((p.relative_to(root).as_posix(), p.read_text()) for p in Path(root).rglob('.done'))


def run(args):
    """Runs a program and stops the cell if it fails, which `!` does not."""
    subprocess.run(args, check=True)


# --- Recordings ---

def load_16k(path):
    """Any audio file as 16 kHz mono 16-bit samples, the device's format."""
    raw = subprocess.run(['ffmpeg', '-v', 'error', '-i', str(path), '-ac', '1', '-ar', '16000',
                          '-f', 's16le', '-'], check=True, capture_output=True).stdout
    return np.frombuffer(raw, dtype=np.int16)


def frame_levels(audio, rate=16000):
    """The level of each 20 ms frame, in dB above a sample value of 1."""
    frame = rate // 50
    count = len(audio) // frame
    frames = audio[:count * frame].astype(np.float64).reshape(count, frame)
    return 10 * np.log10(np.mean(frames ** 2, axis=1) + 1.0), frame


def cut_phrases(audio, single=False):
    """The phrases in a recording, each trimmed to its speech plus a little
    either side. microWakeWord trains on the end of each clip, so silence
    left after a phrase would be learned as the phrase. The background level
    is the quietest tenth of the recording, leaving out digital silence.
    With `single`, everything spoken is kept as one phrase."""
    level, frame = frame_levels(audio)
    sound = level[level > CUT['silence_db']]
    if len(sound) == 0:
        return []
    loud = np.flatnonzero(level > np.percentile(sound, 10) + CUT['above_db'])
    if len(loud) == 0:
        return []
    if single:
        starts, ends = loud[:1], loud[-1:]
    else:
        # Loud frames closer together than the gap belong to one phrase
        breaks = np.flatnonzero(np.diff(loud) > CUT['gap_s'] * 50)
        starts = np.concatenate((loud[:1], loud[breaks + 1]))
        ends = np.concatenate((loud[breaks], loud[-1:]))
    pad = int(CUT['pad_s'] * 50)
    phrases = []
    for start, end in zip(starts, ends):
        if (end - start + 1) / 50 < CUT['min_phrase_s']:
            continue                                    # a click or a knock
        phrases.append(audio[max(0, start - pad) * frame:min(len(level), end + 1 + pad) * frame])
    return phrases


def prepare_recordings(sub, recursive=True):
    """Converts the recordings in RECORDINGS/sub to 16 kHz and cuts them into
    single phrases, one .wav each, in CLIPS_16K/sub, named after the file
    they came from. Returns that directory, or None if there are no
    recordings there. Recordings in Drive are only read, never changed."""
    source = Path(RECORDINGS, sub)
    pattern = source.rglob('*') if recursive else source.glob('*')
    files = sorted(p for p in pattern if p.is_file() and p.suffix.lower() in AUDIO_TYPES) \
        if source.is_dir() else []
    out = Path(CLIPS_16K, sub)
    if not files:
        shutil.rmtree(out, ignore_errors=True)
        return None
    settings = {'files': [(p.relative_to(source).as_posix(), p.stat().st_size) for p in files], 'cut': CUT}
    if step_done(out, settings):
        return out
    fresh(out)
    long_phrases = 0
    for k, path in enumerate(files):
        # Numbered first, so that two files differing only in type stay apart
        stem = f"{k:03d}_{path.relative_to(source).with_suffix('').as_posix().replace('/', '__')}"
        audio = load_16k(path)
        phrases = cut_phrases(audio, single=len(audio) < CUT['min_take_s'] * 16000)
        for j, phrase in enumerate(phrases):
            scipy.io.wavfile.write(out / f'{stem}_{j:02d}.wav', 16000, phrase)
            long_phrases += len(phrase) > 3 * 16000
        print(f'  {path.relative_to(source).as_posix()}: '
              + (f'{len(phrases)} phrase(s)' if phrases else 'no speech found'))
    if long_phrases:
        print(f'  {long_phrases} phrase(s) longer than 3 s: two phrases with too short a pause between them?')
    mark_done(out, settings)
    return out


def test_sets():
    """The test set's folders, as (name, folder under RECORDINGS, holds the
    wake phrase, includes subfolders). Each subfolder of test/hey_claude/
    and test/not_wake/ is reported separately, such as hey_claude/1m;
    recordings directly inside either folder form a set of its own."""
    found = []
    for kind in ('hey_claude', 'not_wake'):
        root = Path(RECORDINGS, 'test', kind)
        if not root.is_dir():
            continue
        if any(p.is_file() for p in root.iterdir()):
            found.append((kind, f'test/{kind}', kind == 'hey_claude', False))
        for sub in sorted(p.name for p in root.iterdir() if p.is_dir()):
            found.append((f'{kind}/{sub}', f'test/{kind}/{sub}', kind == 'hey_claude', True))
    return found


def report_cutoff():
    """The lowest cutoff in the notebook's test report with fewer than
    TARGET_FAPH false detections per hour: the rule docs/WAKEWORD-TRAINING.md
    gives for the manifest's starting cutoff."""
    path = f'{TRAIN_DIR}/tflite_stream_state_internal_quant/tflite_streaming_roc.txt'
    cutoffs = [float(m[1]) for m in re.finditer(r'Cutoff ([\d.]+): frr=[\d.]+; faph=([\d.]+)', open(path).read())
               if float(m[2]) < TARGET_FAPH]
    return min(cutoffs) if cutoffs else None

## Voices

In [ ]:
# Downloads the synthetic voices listed in Settings.

import urllib.request

os.makedirs('voices', exist_ok=True)

def fetch(url, path):
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path + '.part')
        os.rename(path + '.part', path)     # a broken download leaves no file

fetch('https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
      GENERATOR_VOICE)
# Its settings file is not in the release; it is in the source cloned in Install
shutil.copy('/content/piper-sample-generator/models/en_US-libritts_r-medium.pt.json', 'voices/')

HF = 'https://huggingface.co/rhasspy/piper-voices/resolve/main'
for path in VOICES.values():
    base = path.split('/')[-1]
    fetch(f'{HF}/{path}.onnx', f'voices/{base}.onnx')
    fetch(f'{HF}/{path}.onnx.json', f'voices/{base}.onnx.json')

## Listen

In [ ]:
# Listen before generating thousands. Each line plays one sample. If a voice
# mispronounces the phrase, try another spelling in Settings (for example
# 'hey clawd'), or drop that voice from SAMPLE_SETS there.

from IPython.display import Audio, display

def sample(text, model, out):
    # The generator's messages are shown, so a failure explains itself
    !python3 -m piper_sample_generator "{text}" --model {model} --max-samples 1 --output-dir {out}
    wav = f'{out}/0.wav'
    if os.path.exists(wav):
        display(Audio(filename=wav))
    else:
        print(f'No sample from {model}: see the messages above.')

sample(PHRASE_EN, GENERATOR_VOICE, 'check/generator')
sample(PHRASE_EN, voice('l2arctic'), 'check/l2arctic')
sample(PHRASE_EN, voice('vctk'), 'check/vctk')
sample(PHRASE_RU, voice('ru_irina'), 'check/ru')

## Generate samples

In [ ]:
# Generates the sample sets listed in Settings, into Drive. A set made with
# the same settings in an earlier session is kept.

# Version 1 of this notebook made sets without markers. A set without one is
# accepted if it holds enough samples and its settings are exactly those
# version 1 used, which these fingerprints identify; otherwise it is made
# again.
VERSION_1_SETS = {
    'positive/generator': '895342da73fb5cbc',
    'positive/l2arctic': 'd29f7447a7ac8dcb',
    'positive/vctk': 'cab24d3d77534c69',
    'positive/ru': 'c88c2fff768291b3',
    'negative/generator': 'fa2ccb76c577e5d0',
    'negative/l2arctic': '1e28739ac20889c8',
}

def generate(name, phrases, models, count, batch=1, extra=''):
    out = f'{SAMPLES}/{name}'
    settings = {'phrases': phrases, 'models': models, 'count': count, 'speeds': SPEEDS, 'extra': extra}
    if step_done(out, settings):
        print(f'{name}: already generated')
        return
    if (not os.path.exists(f'{out}/.done') and fingerprint(settings) == VERSION_1_SETS.get(name)
            and os.path.isdir(out) and len(list(Path(out).glob('*.wav'))) >= count):
        mark_done(out, settings)
        print(f'{name}: samples from version 1 of this notebook accepted')
        return
    fresh(out)
    # A list of phrases goes to the generator as a file, one per line, which
    # it cycles through
    if isinstance(phrases, list):
        text = f'/content/{name.replace("/", "_")}.txt'
        with open(text, 'w') as f:
            f.write('\n'.join(phrases))
    else:
        text = phrases
    model_args = ' '.join(f'--model {m}' for m in models)
    !python3 -m piper_sample_generator "{text}" {model_args} --max-samples {count} \
        --batch-size {batch} --length-scales {SPEEDS} --output-dir {out} {extra}
    made = len(list(Path(out).glob('*.wav')))
    if made < count:
        raise RuntimeError(f'{name}: {made} of {count} samples made; see the messages above')
    mark_done(out, settings)
    print(f'{name}: {made} samples')

for name, spec in SAMPLE_SETS.items():
    generate(name, **spec)

## Recordings

In [ ]:
# Real voices, if any have been put in Drive following
# docs/WAKEWORD-RECORDING.md:
#   recordings/hey_claude/          the wake phrase, for training
#   recordings/not_wake/            phrases that must not trigger, for training
#   recordings/test/hey_claude/...  the test set, never trained on, one
#   recordings/test/not_wake/...    subfolder per distance or kind
# Without recordings, the run trains on synthetic voices only.
# Check that the number of phrases found in each file matches what was said,
# and listen to the phrases played at the end: each should be one whole
# phrase. The players keep the audio in the notebook's outputs (see the
# privacy note at the top).

import random
from IPython.display import Audio, display

found = {}
for sub in ('hey_claude', 'not_wake'):
    print(f'{sub}:')
    out = prepare_recordings(sub)
    found[sub] = sorted(out.glob('*.wav')) if out else []
    print(f'  {len(found[sub])} phrases' if out else '  none')

tests = test_sets()
for name, sub, _, recursive in tests:
    print(f'test/{name}:')
    out = prepare_recordings(sub, recursive)
    print(f'  {len(list(out.glob("*.wav")))} phrases' if out else '  none')
if found['hey_claude'] and not tests:
    print('No test set: the run cannot be judged on real voices (see docs/WAKEWORD-RECORDING.md).')

for sub, clips in found.items():
    for path in random.sample(clips, min(3, len(clips))):
        print(sub)
        display(Audio(filename=str(path)))

## Background sound

In [ ]:
# Background sound for augmentation: room echoes (MIT impulse responses),
# everyday sounds (one part of AudioSet) and music (FMA), as listed in
# Settings. Adapted from microWakeWord's notebook. Mixed licences: see the
# note at the top. Slow. Files are numbered rather than named after their
# source: current versions of the `datasets` library no longer report a
# clip's original path.

import datasets
from tqdm import tqdm

def save_16k(dataset, output_dir):
    # Resampled to 16 kHz, the rate of the device's microphone
    dataset = dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
    count = 0
    for i, row in enumerate(tqdm(dataset)):
        # Decoded and resampled audio can stray just past +-1.0. Clipped,
        # because such a sample would otherwise wrap round to a full-scale
        # click of the opposite sign.
        audio = np.clip(row['audio']['array'], -1.0, 1.0)
        scipy.io.wavfile.write(os.path.join(output_dir, f'{i}.wav'), 16000,
                               (audio * 32767).astype(np.int16))
        count += 1
    # A download that produced nothing is a failure, not a finished step
    if count == 0:
        raise RuntimeError(f'{output_dir}: no audio was downloaded')

for name, source in BACKGROUND_SOURCES.items():
    if step_done(name, source):
        continue
    fresh(name)
    if 'url' in source:
        # A zip of mp3 files
        fresh('download')
        run(['wget', '-q', '-O', 'download/audio.zip', source['url']])
        run(['unzip', '-q', '-o', 'download/audio.zip', '-d', 'download'])
        dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('download').glob('**/*.mp3')]})
    elif 'data_files' in source:
        # Stored as parquet files, which `datasets` reads directly
        dataset = datasets.load_dataset(source['dataset'], data_files={'train': source['data_files']},
                                        split='train').select_columns(['audio'])
    else:
        dataset = datasets.load_dataset(source['dataset'], split='train', streaming=True)
    save_16k(dataset, name)
    shutil.rmtree('download', ignore_errors=True)
    mark_done(name, source)

print('Background sound ready:', {d: len(os.listdir(d)) - 1 for d in BACKGROUND_SOURCES})

## Augmentation

In [ ]:
# Sets up the augmentation described in Settings, and the clips it is applied
# to. Recordings get the same augmentation without room echo.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.audio_utils import save_clip
from IPython.display import Audio, display

BACKGROUND = ['fma_16k', 'audioset_16k']
augmenter = Augmentation(**AUGMENT, impulse_paths=['mit_rirs'], background_paths=BACKGROUND)
# Without impulse_paths, Augmentation applies no echo
recording_augmenter = Augmentation(**AUGMENT, background_paths=BACKGROUND)

def sample_clips(subdir):
    # Every .wav under the directory, from all voices; 10 % held back for
    # validation and testing
    return Clips(input_directory=f'{SAMPLES}/{subdir}', file_pattern='**/*.wav',
                 max_clip_duration_s=None, remove_silence=False,
                 random_split_seed=10, split_count=0.1)

def recording_clips(sub):
    # All of them, for training: the recordings' test is the separate test set
    folder = Path(CLIPS_16K, sub)
    if not folder.is_dir() or not any(folder.glob('*.wav')):
        return None
    return Clips(input_directory=str(folder), file_pattern='*.wav',
                 max_clip_duration_s=None, remove_silence=False)

positive_clips = sample_clips('positive')
negative_clips = sample_clips('negative')
recordings = {sub: recording_clips(sub) for sub in ('hey_claude', 'not_wake')}

# Listen to one augmented sample of each kind
save_clip(augmenter.augment_clip(positive_clips.get_random_clip()), 'augmented_sample.wav')
display(Audio(filename='augmented_sample.wav'))
if recordings['hey_claude']:
    save_clip(recording_augmenter.augment_clip(recordings['hey_claude'].get_random_clip()),
              'augmented_recording.wav')
    display(Audio(filename='augmented_recording.wav'))

## Features

In [ ]:
# Turns the augmented clips into the spectrogram features the model trains
# on: the same 40-band features the device computes. Each set is rebuilt when
# the clips it is made from or the augmentation settings change.

from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.spectrograms import SpectrogramGeneration

# (directory, Clips split, copies of each clip, slide): training reuses each
# spectrogram shifted by one step at a time, imitating the streaming model;
# testing runs the streaming model itself, so needs none. microWakeWord's
# notebook also slides the validation set; here it does not, because the ten
# copies made validation run out of memory on the free plan (see Negative
# examples), and every held-back sample is still validated once.
SAMPLE_SPLITS = (('training', 'train', 2, 10), ('validation', 'validation', 1, 1), ('testing', 'test', 1, 1))
RECORDING_SPLITS = (('training', None, RECORDING['copies'], 10),)

def make_features(name, clips, augmenter, splits, settings):
    for split, split_name, copies, slide in splits:
        out_dir = f'{FEATURES}/{name}/{split}'
        step = {**settings, 'split': split, 'copies': copies, 'slide': slide, 'step_ms': STEP_MS}
        if step_done(out_dir, step):
            continue
        fresh(out_dir)
        spectrograms = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                             slide_frames=slide, step_ms=STEP_MS)
        RaggedMmap.from_generator(out_dir=f'{out_dir}/wakeword_mmap',
                                  sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=copies),
                                  batch_size=100, verbose=True)
        mark_done(out_dir, step)

make_features('positive', positive_clips, augmenter, SAMPLE_SPLITS,
              {'sources': markers(f'{SAMPLES}/positive'), 'augment': AUGMENT, 'echo': True})
make_features('sound_alikes', negative_clips, augmenter, SAMPLE_SPLITS,
              {'sources': markers(f'{SAMPLES}/negative'), 'augment': AUGMENT, 'echo': True})
for sub, name in (('hey_claude', 'recordings_wake'), ('not_wake', 'recordings_not_wake')):
    if recordings[sub]:
        make_features(name, recordings[sub], recording_augmenter, RECORDING_SPLITS,
                      {'sources': markers(f'{CLIPS_16K}/{sub}'), 'augment': AUGMENT, 'echo': False})
    else:
        # Recordings removed since an earlier build are not trained on
        shutil.rmtree(f'{FEATURES}/{name}', ignore_errors=True)

## Negative examples

In [ ]:
# Negative examples prepared by microWakeWord's author: general speech, a
# dinner party, and sound without speech, already as features. Slow.
#
# Memory. For each validation check, microWakeWord loads the validation sets
# into memory whole and copies them twice more while shuffling. The free
# plan's 12.7 GB is not enough for all of it, and training is killed at the
# first check (step 500). Two reductions:
#
# 1. The sets in NEGATIVE_TRAINING_ONLY are used for training only, not for
#    validation or testing. Validation still covers recall and sound-alikes
#    through this notebook's own held-back samples. Their validation and
#    testing parts are renamed, not deleted, so the change can be undone.

os.makedirs('negative_datasets', exist_ok=True)
for name in ('dinner_party', 'dinner_party_eval', 'no_speech', 'speech'):
    target = f'negative_datasets/{name}'
    settings = {'url': f'{NEGATIVE_URL}/{name}.zip', 'training_only': name in NEGATIVE_TRAINING_ONLY}
    if step_done(target, settings):
        continue
    fresh(target)
    run(['wget', '-q', '-O', f'{target}.zip', settings['url']])
    run(['unzip', '-q', '-o', f'{target}.zip', '-d', 'negative_datasets'])
    # The zip is no longer needed once unpacked, and the disk is not large
    os.remove(f'{target}.zip')
    if settings['training_only']:
        for split in ('validation', 'testing'):
            if os.path.exists(f'{target}/{split}'):
                os.rename(f'{target}/{split}', f'{target}/{split}_unused')
    mark_done(target, settings)

# 2. The dinner-party recording that false detections per hour are measured
#    on (its validation part) is about 9.7 hours, cut into 116,000 overlapping
#    windows (one every 300 ms): 3.8 GB, and over 11 GB with the copies.
#    Measured on a free T4 session: training holds about 3.9 GB, and a
#    validation share of 40 % added 6 GB, which was too much. Validation
#    therefore uses VALIDATION_SHARE (15 %, about 1.5 hours), about 3 GB in
#    all. microWakeWord computes the hours from what is present, so the rate
#    stays correct but coarse: one false detection in 1.5 hours reads as about
#    0.7 per hour, so a checkpoint meets the 0.5 per hour target only with
#    none. The final test runs on the separate test part, about 5.3 hours,
#    which it processes piece by piece without the large copies.
SMALL = 'negative_datasets/dinner_party_eval_small'
small_settings = {'share': VALIDATION_SHARE, 'source': markers('negative_datasets/dinner_party_eval')}
if not step_done(SMALL, small_settings):
    # Emptied whole: an interrupted attempt may have left the link below
    fresh(SMALL)
    os.makedirs(f'{SMALL}/validation_ambient')
    source = [RaggedMmap(str(m)) for m in Path('negative_datasets/dinner_party_eval/validation_ambient').glob('**/*_mmap')]
    clips = [m[i] for m in source for i in range(len(m))]
    budget = int(VALIDATION_SHARE * sum(c.shape[0] for c in clips))
    # The recording is stored as a few clips of about 50 minutes each, so the
    # last one taken is cut to fit; a cut clip is a shorter stretch of the
    # same recording
    kept, used = [], 0
    for clip in clips:
        take = min(clip.shape[0], budget - used)
        if take <= 0:
            break
        kept.append(np.array(clip[:take]))
        used += take
    RaggedMmap.from_generator(out_dir=f'{SMALL}/validation_ambient/part_mmap',
                              sample_generator=iter(kept), batch_size=10, verbose=False)
    # The final test uses the recording's whole test part, by reference
    os.symlink(os.path.abspath('negative_datasets/dinner_party_eval/testing_ambient'),
               f'{SMALL}/testing_ambient')
    mark_done(SMALL, small_settings)
    print(f'Validation recording: {len(kept)} piece(s), {used / 100 / 3600:.2f} hours')

## Train

In [ ]:
# Trains with the settings in Settings, then converts the result to the
# quantised streaming model the device runs, and tests it.
#
# Restart the session first (Runtime -> Restart session), then run Settings,
# Helpers and this cell. The notebook itself holds about 2 GB after
# Augmentation and Features, which training needs; it reads everything it
# uses from disk.
#
# Training runs as a separate process with its log in train.log. This cell
# prints the process's memory whenever it changes by 0.5 GB, then how the
# process ended and the end of the log. Exit code 0 is success; -9 means
# Colab stopped it for using too much memory (see Negative examples). Long
# silences are normal: memory stays flat while training.

import subprocess, time, psutil, yaml

def features(path, weight, truth, strategy, splits, penalty=1.0):
    # Every part the set should have must be complete
    missing = [s for s in splits if not Path(path, s, '.done').exists()]
    if missing:
        raise RuntimeError(f'{path}: {", ".join(missing)} missing; run the sections from Settings '
                           'to Negative examples first')
    return {'features_dir': path, 'sampling_weight': weight, 'penalty_weight': penalty,
            'truth': truth, 'truncation_strategy': strategy, 'type': 'mmap'}

# The parts each set must have; a downloaded set has one marker of its own
ALL_SPLITS, WHOLE_SET = ('training', 'validation', 'testing'), ('',)
feature_sets = [
    features(f'{FEATURES}/positive', SET_WEIGHTS['positive'], True, 'truncate_start', ALL_SPLITS),
    features(f'{FEATURES}/sound_alikes', SET_WEIGHTS['sound_alikes'], False, 'random', ALL_SPLITS,
             penalty=SOUND_ALIKE_PENALTY),
    *[features(f'negative_datasets/{n}', SET_WEIGHTS[n], False, 'random', WHOLE_SET)
      for n in ('speech', 'dinner_party', 'no_speech')],
    # Validation and testing only: estimates false detections per hour.
    # The reduced copy made in Negative examples; see the memory note there.
    features('negative_datasets/dinner_party_eval_small', 0.0, False, 'split', WHOLE_SET),
]
# A set is drawn from in proportion to its weight, whatever its size, so the
# recordings' weight sets their share of the examples they stand beside
share = RECORDING['share'] / (1 - RECORDING['share'])
if os.path.isdir(f'{FEATURES}/recordings_wake'):
    feature_sets.append(features(f'{FEATURES}/recordings_wake', SET_WEIGHTS['positive'] * share,
                                 True, 'truncate_start', ('training',)))
if os.path.isdir(f'{FEATURES}/recordings_not_wake'):
    feature_sets.append(features(f'{FEATURES}/recordings_not_wake', SET_WEIGHTS['sound_alikes'] * share,
                                 False, 'random', ('training',), penalty=SOUND_ALIKE_PENALTY))
print('Training on:', ', '.join(f['features_dir'] for f in feature_sets if f['sampling_weight'] > 0))

config = {
    **TRAINING,
    'window_step_ms': STEP_MS,
    'train_dir': TRAIN_DIR,
    'features': feature_sets,
    'target_minimization': TARGET_FAPH,
    'minimization_metric': 'ambient_false_positives_per_hour',
    'maximization_metric': 'average_viable_recall',
}
# Training resumes from the run's last checkpoint, so a run continued with
# different settings or features would mix them in one model. The run keeps
# a fingerprint of what it was started with, and refuses to continue with
# anything else.
os.makedirs(TRAIN_DIR, exist_ok=True)
started = {'config': config, 'model': MODEL_ARGS, 'features': [markers(f['features_dir']) for f in feature_sets]}
if Path(TRAIN_DIR, '.started').exists():
    if Path(TRAIN_DIR, '.started').read_text().strip() != fingerprint(started):
        raise RuntimeError(f'Run {RUN} was started with other settings or features. Set a new RUN in '
                           'Settings to train with these, or restore the earlier ones to continue it.')
else:
    Path(TRAIN_DIR, '.started').write_text(fingerprint(started))
# Kept with the run in Drive, so the run's settings stay with its results
config_path = f'{TRAIN_DIR}/training_parameters.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f)

args = ['python', '-m', 'microwakeword.model_train_eval',
        f'--training_config={config_path}', '--train', '1', '--restore_checkpoint', '1',
        '--test_tf_nonstreaming', '0', '--test_tflite_nonstreaming', '0',
        '--test_tflite_nonstreaming_quantized', '0', '--test_tflite_streaming', '0',
        '--test_tflite_streaming_quantized', '1', '--use_weights', 'best_weights', *MODEL_ARGS]
with open('train.log', 'w') as log:
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    p, peak, last = psutil.Process(proc.pid), 0, 0
    while proc.poll() is None:
        try:
            rss = p.memory_info().rss / 2**30   # bytes to GB
        except psutil.NoSuchProcess:
            break
        peak = max(peak, rss)
        if abs(rss - last) > 0.5:
            print(f'{time.strftime("%H:%M:%S")}  training process: {rss:.1f} GB')
            last = rss
        time.sleep(2)
exit_code = proc.wait()
print('exit code:', exit_code, ' peak:', f'{peak:.1f} GB')
!tail -40 train.log
if exit_code != 0:
    # Stops here, so that the next sections do not test or keep an earlier model
    raise RuntimeError(f'Training failed (exit code {exit_code}); see the log above')

## Test on real voices

In [ ]:
# Runs the finished model on the test recordings and decides as the device
# does (firmware/components/wakeword/wakeword.cc): features scaled to the
# model's input in the same integer arithmetic, and a detection when the sum
# of the last WINDOW scores, each 0-255, exceeds the cutoff's 0-255 form
# times WINDOW. The notebook's own test, on synthetic samples, has proved a
# poor guide to the device; this one is the run's measure on real voices.
#
# Any run's model can be tested, including earlier ones: set RUN in Settings
# to that run, and run Settings, Helpers and this cell.

from numpy.lib.stride_tricks import sliding_window_view
from microwakeword.inference import Model
from microwakeword.audio.audio_utils import generate_features_for_clip

MODEL_PATH = f'{TRAIN_DIR}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
# On the device, a phrase is heard in a stream of the room's background:
# the noise reduction has long settled on it, and the stream carries on
# after the phrase while the model decides, which takes up to about 0.4 s
# after the speech ends. Each clip is therefore placed between LEAD_S and
# TAIL_S of that background, made from the clip's own quietest stretches.
LEAD_S, TAIL_S = 2.0, 1.0

def to_u8(p):
    # The firmware's rounding of a 0-1 value to 0-255
    return int(p * 255 + 0.5)

def device_input(features):
    # microWakeWord's features are the frontend's values divided by 25.6; the
    # firmware turns those values into the model's int8 input as
    # value * 256 / 666 - 128, rounded and clamped
    values = np.rint(features * 25.6).astype(np.int64)
    return np.clip((values * 256 + 333) // 666 - 128, -128, 127).astype(np.int8)

def background(audio, seconds):
    # The quietest quarter of the clip's 20 ms frames, repeated to length
    level, frame = frame_levels(audio)
    quiet = np.argsort(level)[:max(1, len(level) // 4)]
    sound = np.concatenate([audio[i * frame:(i + 1) * frame] for i in sorted(quiet)])
    return np.resize(sound, int(seconds * 16000))

def peak_sum(path):
    """The highest sum of WINDOW consecutive 0-255 scores for a clip."""
    _, audio = scipy.io.wavfile.read(path)
    audio = np.concatenate([background(audio, LEAD_S), audio, background(audio, TAIL_S)])
    features = device_input(generate_features_for_clip(audio, STEP_MS))
    # A fresh model per clip, so that no clip's state carries into the next
    scores = [to_u8(p) for p in Model(MODEL_PATH).predict_spectrogram(features)]
    if len(scores) < WINDOW:
        return 0
    return int(sliding_window_view(np.array(scores), WINDOW).sum(axis=-1).max())

peaks, wake = {}, {}
for name, sub, is_wake, recursive in test_sets():
    folder = prepare_recordings(sub, recursive)
    if folder:
        peaks[name] = {p.stem: peak_sum(p) for p in sorted(folder.glob('*.wav'))}
        wake[name] = is_wake

if not peaks:
    print('No test recordings in Drive: nothing to test.')
else:
    start = report_cutoff()
    test_id = fingerprint(markers(f'{CLIPS_16K}/test'))
    cutoffs = sorted({round(c, 2) for c in np.arange(0.30, 0.95, 0.05)} | ({start} if start else set()))
    lines = [f'Real-voice test of {RUN}, on test set {test_id}.',
             'Detections in each test folder at each cutoff, decided as the device decides. For',
             'not_wake folders, the count is of false detections.',
             f'Manifest starting cutoff (lowest under {TARGET_FAPH} false detections per hour): {start}', '']
    lines.append('cutoff  ' + '  '.join(f'{n:>16}' for n in peaks))
    for c in cutoffs:
        row = '  '.join(f'{sum(s > to_u8(c) * WINDOW for s in v.values()):>9} of {len(v):<3}'
                        for v in peaks.values())
        lines.append(f'{c:5.2f}{" *" if c == start else "  "} {row}')
    lines += ['', 'Each recording\'s highest averaged score, highest first:']
    for name, scores in peaks.items():
        lines.append(f'{name}:')
        lines += [f'  {s / (255 * WINDOW):.3f}  {clip}' for clip, s in sorted(scores.items(), key=lambda kv: -kv[1])]
    report = '\n'.join(lines)
    print(report)
    with open(f'{TRAIN_DIR}/real_voice_test.txt', 'w', encoding='utf-8') as f:
        f.write(report + '\n')

## Keep results

In [ ]:
# Keeps the results in Drive and downloads them: the model; the notebook's
# test report, which the cutoff is chosen from; the real-voice test, if any;
# the firmware manifest; and a record of the run's settings. Names carry the
# run, as the firmware's model directory expects (docs/WAKEWORD-TRAINING.md).

from google.colab import files

name = f'hey_claude_{RUN}'
results = f'{TRAIN_DIR}/tflite_stream_state_internal_quant'
if not os.path.exists(f'{results}/stream_state_internal_quant.tflite'):
    raise RuntimeError(f'No model in {results}: has Train finished?')
kept = {
    f'{name}.tflite': f'{results}/stream_state_internal_quant.tflite',
    f'{name}_report.txt': f'{results}/tflite_streaming_roc.txt',
    f'{name}_real_voice_test.txt': f'{TRAIN_DIR}/real_voice_test.txt',
}
for target, source in kept.items():
    if os.path.exists(source):
        shutil.copy(source, f'{KEEP}/{target}')

# The firmware manifest. The cutoff follows the rule in the training
# document; on-device tuning adjusts it.
cutoff = report_cutoff()
manifest = {
    'type': 'micro',
    'wake_word': 'Hey Claude',
    'author': 'Hey Claude project',
    'model': f'{name}.tflite',
    'trained_languages': ['en'],
    'version': 2,
    'micro': {
        'probability_cutoff': cutoff,
        'feature_step_size': STEP_MS,
        'sliding_window_size': WINDOW,
        'tensor_arena_size': ARENA,
        'minimum_esphome_version': '2024.7.0',
    },
}
with open(f'{KEEP}/{name}.json', 'w') as f:
    json.dump(manifest, f, indent=2)
    f.write('\n')
if cutoff is None:
    print(f'No cutoff in the report stays under {TARGET_FAPH} false detections per hour: set one by hand.')

# What the run was made with: every value in Settings, the notebook and the
# tools. Recordings are counted by kind only, since file and folder names
# may contain people's names.
def wav_count(folder):
    return len(list(Path(folder).rglob('*.wav'))) if Path(folder).is_dir() else 0

record = {
    'notebook': NOTEBOOK,
    'run': RUN,
    'microwakeword_commit': subprocess.run(['git', '-C', '/content/microWakeWord', 'rev-parse', 'HEAD'],
                                           capture_output=True, text=True).stdout.strip(),
    'phrase_en': PHRASE_EN,
    'phrase_ru': PHRASE_RU,
    'sound_alikes': SOUND_ALIKES,
    'voices': VOICES,
    'speeds': SPEEDS,
    'sample_sets': SAMPLE_SETS,
    'samples_made': {n: wav_count(f'{SAMPLES}/{n}') for n in SAMPLE_SETS},
    'augmentation': AUGMENT,
    'background_sources': BACKGROUND_SOURCES,
    'recording': RECORDING,
    'cut': CUT,
    'recordings': {
        'training_wake': wav_count(f'{CLIPS_16K}/hey_claude'),
        'training_not_wake': wav_count(f'{CLIPS_16K}/not_wake'),
        'test_wake': wav_count(f'{CLIPS_16K}/test/hey_claude'),
        'test_not_wake': wav_count(f'{CLIPS_16K}/test/not_wake'),
    },
    'test_set': fingerprint(markers(f'{CLIPS_16K}/test')) if Path(CLIPS_16K, 'test').is_dir() else None,
    'negative_training_only': NEGATIVE_TRAINING_ONLY,
    'validation_share': VALIDATION_SHARE,
    'target_faph': TARGET_FAPH,
    'training': TRAINING,
    'set_weights': SET_WEIGHTS,
    'sound_alike_penalty': SOUND_ALIKE_PENALTY,
    'model_args': MODEL_ARGS,
    'step_ms': STEP_MS,
    'window': WINDOW,
    'arena': ARENA,
    'training_parameters': open(f'{TRAIN_DIR}/training_parameters.yaml').read(),
    'packages': subprocess.run(['pip', 'freeze'], capture_output=True, text=True).stdout.splitlines(),
}
with open(f'{KEEP}/{name}_settings.json', 'w', encoding='utf-8') as f:
    json.dump(record, f, indent=2, ensure_ascii=False)

for target in [*kept, f'{name}.json', f'{name}_settings.json']:
    if os.path.exists(f'{KEEP}/{target}'):
        files.download(f'{KEEP}/{target}')
print(open(f'{results}/tflite_streaming_roc.txt').read())
print('Before saving a copy of this notebook or sharing it: Edit -> Clear all outputs.')